# 03 · Normalisation du dialecte tunisien (extension 1)

**Objectifs**
1. Étudier les variations orthographiques réelles des transcriptions TEDxTN.
2. Définir et **documenter** des conventions de normalisation.
3. Implémenter et comparer quatre méthodes : **règles**, **lexique de formes majoritaires**,
   **seq2seq (ByT5)** et **LLM** ; plus une transformation vers l'**arabe standard** (NLLB).
4. Évaluer : mesures intrinsèques, précision sur des variantes jamais vues, impact sur le WER et la
   traduction, et analyse manuelle structurée.

Ce notebook ne demande que les annotations (pas l'audio). Les parties ByT5 / LLM / NLLB utilisent le GPU.

## 1. Introduction
Le tunisien n'a pas d'orthographe officielle. Les transcripteurs de TEDxTN suivent un guide interne, mais un
même mot reste écrit de plusieurs façons : hamza ou non (أنا / انا / آنا), alif maqsura ou ya finale (على / علي),
allongements expressifs (برشاااا), emprunts en arabe ou en latin (سيركيلاسيون / circulation), article collé
(الfumoir). Normaliser sert à : (a) évaluer l'ASR sans pénaliser des variantes légitimes, (b) réduire la
dispersion du vocabulaire pour les modèles de texte, (c) produire un texte plus cohérent pour la traduction.

Deux conventions sont possibles, comme le demande l'énoncé :
- **tunisien écrit selon des conventions homogènes** (ce notebook, sections 3 à 6) ;
- **transformation vers l'arabe standard** (section 7), qui change la langue et donc perd de l'information.

## 2. Installation et données

In [ ]:
import os, subprocess, sys
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/roudayn-kouka/tedxtn-tunisian-speech.git"
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("TEDXTN_WORKDIR", "/content/drive/MyDrive/tedxtn")
    project = Path("/content/drive/MyDrive/tedxtn-tunisian-speech")
    if not project.exists():
        project = Path("/content/tedxtn-tunisian-speech")
        if not project.exists():
            subprocess.run(["git", "clone", "-q", REPO_URL, str(project)], check=True)
    os.chdir(project)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

In [ ]:
import random, re
from collections import Counter, defaultdict
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import HTML, display
from common import data, metrics, text
from common.config import BASE_MODELS, MODELS_DIR, RESULTS_DIR, SEED, SMOKE, TrainConfig, env_flag, gpu_memory_gb
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
try:
    import torch; HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
VRAM = gpu_memory_gb()
# Interrupteurs forçables sans modifier le notebook : TEDXTN_RUN_<NOM>=0/1
RUN_BYT5, RUN_LLM = env_flag("BYT5", HAS_GPU), env_flag("LLM", HAS_GPU)
RUN_MSA, RUN_MT_IMPACT = env_flag("MSA", HAS_GPU), env_flag("MT_IMPACT", HAS_GPU)
LLM_MODEL = BASE_MODELS["llm"] if VRAM >= 20 else BASE_MODELS["llm-small"]

df = data.load_corpus(download=os.environ.get("TEDXTN_SKIP_DOWNLOAD") != "1")
train, dev, test = (df[df.split == s] for s in ["train", "dev", "test"])
print(len(train), len(dev), len(test))

## 3. Étude des variations orthographiques
On regroupe les mots du corpus qui ont le même **squelette** après suppression des différences purement
graphiques (règles de la section 4). Chaque groupe qui contient plusieurs formes est une variation.
On classe ensuite chaque paire (forme rare → forme majoritaire) selon le phénomène en cause.

In [ ]:
freq = Counter(w for t in df.transcription for w in t.split())
clusters = defaultdict(Counter)
for w, c in freq.items():
    k = text.normalize_text(w, text.WRITING_NORM)   # convention d'écriture (frontière de script comprise)
    if k:
        clusters[k][w] += c
multi = {k: v for k, v in clusters.items() if len(v) > 1}
print(f"{len(freq):,} formes ; {len(clusters):,} squelettes ; {len(multi):,} squelettes avec ≥ 2 graphies "
      f"({sum(sum(v.values()) for v in multi.values()) / sum(freq.values()) * 100:.1f} % des occurrences)")

_OFF = dict(remove_brackets=False, remove_diacritics=False, normalize_alef=False, normalize_ya=False, normalize_gaf=False,
            arabic_digits=False, remove_punctuation=False, lowercase_latin=False, collapse_elongation=False,
            split_mixed_tokens=False, remove_hesitations=False)
SINGLE_RULES = {"hamza sur alef": "normalize_alef", "alif maqsura / ya": "normalize_ya", "diacritiques / tatweel": "remove_diacritics",
                "/g/ (ڨ گ)": "normalize_gaf", "allongement": "collapse_elongation", "casse (latin)": "lowercase_latin",
                "frontière arabe/latin": "split_mixed_tokens", "ponctuation": "remove_punctuation"}

def phenomenon(a, b):
    """Règle(s) qui, appliquée(s) seule(s), rend(ent) les deux graphies identiques."""
    found = [name for name, flag in SINGLE_RULES.items()
             if text.normalize_text(a, text.NormConfig(**{**_OFF, flag: True})) == text.normalize_text(b, text.NormConfig(**{**_OFF, flag: True}))]
    return found or ["combinaison de règles"]

ph = Counter()
for k, v in multi.items():
    top, *rest = [w for w, _ in v.most_common()]
    for r in rest:
        for p in phenomenon(r, top):
            ph[p] += v[r]
ph = pd.Series(ph).sort_values()
fig, ax = plt.subplots(figsize=(7, 3)); ph.plot.barh(ax=ax, color="#c2410c"); ax.set_title("Occurrences en graphie minoritaire, par phénomène")
plt.tight_layout(); plt.savefig(RESULTS_DIR / "03_phenomena.png", bbox_inches="tight"); plt.show()
display(pd.DataFrame([{"squelette": k, "graphies": ", ".join(f"{w} ({c})" for w, c in v.most_common(5))}
                      for k, v in sorted(multi.items(), key=lambda kv: -sum(kv[1].values()))[:15]]))

### 3.1 Emprunts écrits dans deux scripts
On cherche des paires (mot en script arabe, mot en script latin) qui se prononcent pareil, grâce au squelette
phonétique (`text.phonetic_skeleton`). Ces paires forment un **lexique d'emprunts candidat**, exporté pour
validation manuelle (`results/03_loanword_candidates.csv`).

In [ ]:
lat_words = {w.lower(): c for w, c in freq.items() if text.token_script(w) == "lat" and len(w) >= 4 and c >= 3}
ar_words = {w: c for w, c in freq.items() if text.token_script(w) == "ar" and len(w) >= 4 and c >= 2}
by_skel = defaultdict(list)
for w in lat_words:
    s = text.phonetic_skeleton(w)
    if len(s) >= 3:
        by_skel[s].append(w)
cands = []
for a, c in ar_words.items():
    s = text.phonetic_skeleton(a)
    if len(s) >= 3 and s in by_skel:
        for l in by_skel[s]:
            cands.append({"arabe": a, "freq_arabe": c, "latin": l, "freq_latin": lat_words[l], "squelette": s})
cands = pd.DataFrame(cands).sort_values("freq_arabe", ascending=False) if cands else pd.DataFrame(columns=["arabe", "latin"])
cands["valide (o/n)"] = ""
cands.to_csv(RESULTS_DIR / "03_loanword_candidates.csv", index=False)
print(len(cands), "paires candidates"); display(cands.head(15))

## 4. Conventions retenues

| # | Convention | Exemple | Justification |
|---|---|---|---|
| 1 | Supprimer diacritiques et tatweel | مَشِيت → مشيت | rares, incohérents dans le corpus |
| 2 | Alef sans hamza | أنا, آنا → انا | la hamza initiale n'est pas prononcée distinctement en tunisien |
| 3 | ya finale | على → علي | confusion graphique fréquente, même prononciation |
| 4 | /g/ tunisien unifié en ڨ | گاز → ڨاز | trois lettres pour un seul son |
| 5 | Chiffres arabes occidentaux | ٣ → 3 | une seule forme |
| 6 | Ponctuation supprimée, latin en minuscules | Hôtel → hôtel | l'oral n'a pas de ponctuation fiable |
| 7 | Allongements réduits | برشاااا → برشا | marque expressive, pas lexicale |
| 8 | Article séparé d'un mot latin | الfumoir → ال fumoir | rend le mot étranger reconnaissable |
| 9 | (option) emprunts en latin via lexique validé | سيركيلاسيون → circulation | un mot, une graphie |
| — | **Non retenu** : ة → ه, ؤ/ئ → و/ي | | ces distinctions portent parfois du sens |

Ces conventions forment `WRITING_NORM` (convention d'écriture, utilisée ici et par l'interface). L'évaluation de
l'ASR n'en garde que la partie orthographique, `EVAL_NORM` (conventions 1 à 7) : la convention 8 est asymétrique en
évaluation (les références écrivent les emprunts en arabe, seules les hypothèses contiennent des mots hybrides).
Le WER sans hésitations (`EVAL_NORM_NOHES` : آ, آم, إم, أ, إ, أه, اه, euh, ممم...) est rapporté à part ; les
hésitations sont repérées sur la forme d'origine du mot, avant la normalisation de l'alef (« أم » est un mot).
Ces règles sont dans `common/text.py` (`NormConfig`) et partagées avec les notebooks 01-02 et l'interface.

**Deuxième convention, fondée sur les données** : pour chaque squelette, la **graphie majoritaire** du train
devient la forme canonique. Contrairement aux règles, elle garde la hamza ou la ta marbuta quand l'usage
majoritaire les garde, et ne détruit donc pas d'information.

## 5. Méthodes
### 5.1 Règles (convention 1), avec ablation règle par règle

In [ ]:
RULES = text.WRITING_NORM   # convention d'écriture : conventions 1 à 8 (frontière de script comprise)
def vocab_stats(texts, fn):
    toks = [w for t in texts for w in fn(t).split()]
    return len(set(toks)), len(toks)

base_v, base_n = vocab_stats(train.transcription, lambda t: t)
rows = [{"configuration": "aucune (brut)", "vocabulaire": base_v, "mots": base_n}]
flags = ["remove_diacritics", "normalize_alef", "normalize_ya", "normalize_gaf", "arabic_digits", "remove_punctuation",
         "lowercase_latin", "collapse_elongation", "split_mixed_tokens"]
for f in flags:  # ajout cumulatif des règles
    cfg = text.NormConfig(**{g: (g in flags[:flags.index(f) + 1]) for g in flags}, remove_brackets=True, remove_hesitations=False)
    v, n = vocab_stats(train.transcription, lambda t, c=cfg: text.normalize_text(t, c))
    rows.append({"configuration": f"+ {f}", "vocabulaire": v, "mots": n})
abl = pd.DataFrame(rows); abl["réduction vocab. (%)"] = (100 * (1 - abl.vocabulaire / base_v)).round(2)
display(abl)

### 5.2 Lexique de formes majoritaires (convention 2)
Construit **sur le train uniquement**. Un mot inconnu du lexique est laissé tel quel.

In [ ]:
def build_majority_lexicon(texts):
    cl = defaultdict(Counter)
    for t in texts:
        for w in t.split():
            k = text.normalize_text(w, RULES)
            if k:
                cl[k][w] += 1
    lex = {}
    for k, v in cl.items():
        canon = v.most_common(1)[0][0]
        for w in v:
            lex[w] = canon
    return lex, {k: v.most_common(1)[0][0] for k, v in cl.items()}

lex, canon_by_key = build_majority_lexicon(train.transcription)
def lexicon_normalize(t):
    return " ".join(lex.get(w, w) for w in t.split())
print(len(lex), "formes dans le lexique ;", sum(1 for w, c in lex.items() if w != c), "formes réécrites")

### 5.3 Seq2seq ByT5 (convention 2, généralisation aux mots inconnus)
Le lexique ne sait rien des mots absents du train. Un modèle seq2seq au niveau octet (ByT5-small) peut
apprendre la convention et l'appliquer à des mots jamais vus.
**Données** : phrases du train où chaque mot d'un groupe de variantes est remplacé aléatoirement par l'une de
ses graphies observées (entrée) ; cible = la phrase avec les formes majoritaires.
**Test honnête** : les squelettes sont séparés en deux ; ceux du test ne sont jamais vus à l'entraînement.

In [ ]:
# Groupes de variantes et formes canoniques définis sur le TRAIN uniquement
train_clusters = defaultdict(Counter)
for t in train.transcription:
    for w in t.split():
        k = text.normalize_text(w, RULES)
        if k:
            train_clusters[k][w] += 1
keys = sorted(k for k, v in train_clusters.items() if len(v) > 1)
rng = random.Random(SEED); rng.shuffle(keys)
held_out = set(keys[: max(1, len(keys) // 5)])          # squelettes réservés au test
variants_of = {k: list(train_clusters[k]) for k in keys}
canon = {k: train_clusters[k].most_common(1)[0][0] for k in keys}

def corrupt(sentence, allowed, p=0.6):
    """Remplace aléatoirement les mots des groupes `allowed` par une de leurs graphies observées.
    Renvoie (entrée, cible, positions à normaliser)."""
    src, tgt, pos = [], [], []
    for i, w in enumerate(sentence.split()):
        k = text.normalize_text(w, RULES)
        if k in allowed:
            tgt.append(canon[k]); src.append(rng.choice(variants_of[k]) if rng.random() < p else w); pos.append(i)
        else:
            src.append(w); tgt.append(w)
    return " ".join(src), " ".join(tgt), pos

train_keys = set(keys) - held_out
pairs_train = [corrupt(t, train_keys) for t in train.transcription for _ in range(2)]
pairs_train = [(s, t) for s, t, pos in pairs_train if pos]
pairs_test = [corrupt(t, held_out, p=1.0) for t in pd.concat([dev, test]).transcription]
pairs_test = [x for x in pairs_test if x[2]][:1500]
# Lexique d'évaluation : construit sans les squelettes réservés (ils simulent des mots inconnus)
lex_eval = {w: canon[k] for k in train_keys for w in variants_of[k]}
if SMOKE:   # essai rapide : peu de paires (le texte du corpus est complet, seul l'entraînement est raccourci)
    pairs_train, pairs_test = pairs_train[:400], pairs_test[:100]
print(len(pairs_train), "paires d'entraînement ;", len(pairs_test), "paires de test (squelettes jamais vus)")
display(pd.DataFrame(pairs_train[:5], columns=["entrée (variantes)", "cible (formes majoritaires)"]))

In [ ]:
BYT5_DIR = MODELS_DIR / "norm_byt5"
from common.config import trained
if RUN_BYT5 and trained(BYT5_DIR):
    print("modèle final trouvé, entraînement sauté :", BYT5_DIR / "final")
elif RUN_BYT5:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq
    from common import asr as A
    tok = AutoTokenizer.from_pretrained("google/byt5-small")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-small")      # poids en fp32
    class PairDS(torch.utils.data.Dataset):
        def __init__(self, p): self.p = p
        def __len__(self): return len(self.p)
        def __getitem__(self, i):
            s, t = self.p[i]; return tok(s, text_target=t, truncation=True, max_length=512)
    split = int(.95 * len(pairs_train))   # 5 % des paires du train : validation pour l'early stopping
    # Arguments communs du projet : AdamW 8 bits (TEDXTN_OPTIM), plafond de batch avec accumulation (TEDXTN_MAX_BATCH),
    # early stopping sur la loss de validation. T5 est instable en fp16 : bf16 si la carte le permet, sinon fp32.
    # toutes les paires du train (hors mode SMOKE), et au moins 1 000 steps d'optimisation ;
    # TEDXTN_BYT5_EPOCHS=<n> fixe le nombre d'epochs (GPU local : 1 epoch, ≈ 1 250 steps, contrôle de recopie gardé)
    import math
    epochs = int(os.environ.get("TEDXTN_BYT5_EPOCHS", "0")) or max(3, math.ceil(1000 / math.ceil(split / 16)))
    c = TrainConfig(learning_rate=3e-4, batch_size=16, grad_accum=1, max_epochs=epochs, eval_steps=500, fp16=False,
                    gradient_checkpointing=True, spec_augment=False)
    print(f"ByT5 : {split} paires d'entraînement, {epochs} epochs ≈ {epochs * math.ceil(split / 16)} steps")
    model.config.use_cache = False
    args = A.training_args(BYT5_DIR, c, seq2seq=True, bf16=torch.cuda.is_bf16_supported())
    trainer = A.make_trainer(model, args, PairDS(pairs_train[:split]), PairDS(pairs_train[split:]),
                             DataCollatorForSeq2Seq(tok, model=model), type("P", (), {"feature_extractor": tok})(), c)
    last = max(BYT5_DIR.glob("checkpoint-*"), default=None, key=lambda p: int(p.name.split("-")[1])) if BYT5_DIR.exists() else None
    trainer.train(resume_from_checkpoint=str(last) if last else None)
    trainer.save_model(BYT5_DIR / "final"); tok.save_pretrained(BYT5_DIR / "final")
    pd.DataFrame(trainer.state.log_history).to_csv(RESULTS_DIR / "train_log_norm_byt5.csv", index=False)
    del model, trainer; torch.cuda.empty_cache()

def load_byt5():
    """Recharge ByT5 et vérifie que les embeddings d'entrée de l'encodeur et du décodeur sont liés à `shared`
    (sinon le modèle rechargé n'est pas celui entraîné)."""
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
    t_ = AutoTokenizer.from_pretrained(BYT5_DIR / "final")
    m = AutoModelForSeq2SeqLM.from_pretrained(BYT5_DIR / "final")
    m.tie_weights()
    sh = m.shared.weight.data_ptr()
    assert m.encoder.embed_tokens.weight.data_ptr() == sh and m.decoder.embed_tokens.weight.data_ptr() == sh, \
        "ByT5 : embeddings non liés après rechargement"
    return t_, m.cuda().eval()

def byt5_normalize(texts, bs=32):
    t_, m = load_byt5()
    out = []
    for i in range(0, len(texts), bs):
        enc = t_(texts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        # une normalisation a presque la longueur de son entrée : borne de génération adaptée au lot
        # (512 octets fixes rendaient l'inférence très lente, octet par octet)
        n_new = min(512, int(1.3 * enc["attention_mask"].sum(1).max().item()) + 16)
        with torch.no_grad():
            g = m.generate(**enc, max_new_tokens=n_new, num_beams=2)
        out += t_.batch_decode(g, skip_special_tokens=True)
    del m; torch.cuda.empty_cache()
    return out

### 5.4 LLM few-shot (volet GenAI)
Le LLM reçoit les conventions de la section 4 en consigne et 5 exemples produits par les règles.

In [ ]:
llm_outputs = None
sample_eval = test.sample(min(100, len(test)), random_state=SEED).transcription.tolist()
if RUN_LLM:
    from common import llm as L
    tokL, mL = L.load_llm(LLM_MODEL)
    shots = [(t, text.normalize_text(t, RULES)) for t in train.sample(5, random_state=1).transcription]
    llm_outputs = L.llm_normalize(tokL, mL, sample_eval, shots)
    del mL; torch.cuda.empty_cache()

## 6. Évaluation
### 6.1 Mesures intrinsèques (dev + test)
- **vocabulaire** et **taux de mots inconnus** (OOV) par rapport au train normalisé de la même façon ;
- **conservation des mots** : la normalisation ne doit ni ajouter ni supprimer de mots (ratio de longueur) ;
- **précision sur variantes jamais vues** (convention 2) : part des mots ramenés à la forme majoritaire.

In [ ]:
eval_texts = pd.concat([dev, test]).transcription.tolist()
methods = {"brut": lambda t: t, "règles": lambda t: text.normalize_text(t, RULES), "lexique majoritaire": lexicon_normalize}
rows = []
for name, fn in methods.items():
    tr_vocab = {w for t in train.transcription for w in fn(t).split()}
    toks = [w for t in eval_texts for w in fn(t).split()]
    raw_len = sum(len(t.split()) for t in eval_texts)
    rows.append({"méthode": name, "vocabulaire (dev+test)": len(set(toks)), "OOV (%)": 100 * np.mean([w not in tr_vocab for w in toks]),
                 "ratio de longueur": len(toks) / raw_len})
intr = pd.DataFrame(rows).round(3); display(intr)

In [ ]:
def word_acc(preds, golds, positions):
    """Exactitude sur les seuls mots à normaliser (ceux des squelettes jamais vus)."""
    ok = tot = 0
    for p, g, pos in zip(preds, golds, positions):
        pw, gw = p.split(), g.split()
        for i in pos:
            tot += 1; ok += i < len(pw) and pw[i] == gw[i]
    return 100 * ok / max(tot, 1)

src_t, gold_t, pos_t = [x[0] for x in pairs_test], [x[1] for x in pairs_test], [x[2] for x in pairs_test]
acc = {"identité (ne rien faire)": word_acc(src_t, gold_t, pos_t),
       "lexique majoritaire": word_acc([" ".join(lex_eval.get(w, w) for w in s.split()) for s in src_t], gold_t, pos_t)}
BYT5_OK = None
if RUN_BYT5 or (BYT5_DIR / "final").exists():
    # Contrôle de cohérence : 50 phrases du dev sans aucun mot d'un groupe de variantes doivent être recopiées
    # (quasi) à l'identique. Sinon le modèle est cassé et son score n'est pas rapporté.
    from difflib import SequenceMatcher
    clean = [t for t in dev.transcription if t.split() and not any(text.normalize_text(w, RULES) in variants_of for w in t.split())][:50]
    copy = byt5_normalize(clean)
    exact = 100 * np.mean([a == b for a, b in zip(clean, copy)])
    sim = 100 * np.mean([SequenceMatcher(None, a, b).ratio() for a, b in zip(clean, copy)])
    BYT5_OK = sim >= 95
    print(f"Recopie de {len(clean)} phrases sans variante : {exact:.0f} % identiques, similarité caractères {sim:.1f} % "
          f"-> modèle {'cohérent' if BYT5_OK else 'CASSÉ : score non rapporté'}")
    if BYT5_OK:
        acc["ByT5"] = word_acc(byt5_normalize(src_t), gold_t, pos_t)
acc = pd.Series(acc, name="exactitude (%) sur les mots de squelettes jamais vus").round(2); display(acc)

### 6.2 Impact sur l'évaluation de l'ASR
Les prédictions du notebook 01 sont réévaluées avec chaque normalisation appliquée à la référence **et** à
l'hypothèse. La baisse du WER mesure la part des « erreurs » qui ne sont que des choix d'écriture.

In [ ]:
asr_preds = data.load_preds("asr", "test")
rows = []
for sysname, p in asr_preds.items():
    r, h = p.ref.tolist(), p.hyp.tolist()
    rows.append({"système": sysname,
                 "WER brut": metrics.wer_cer(r, h, text.RAW_NORM)["WER"],
                 "WER normalisé (orthographe)": metrics.wer_cer(r, h, text.EVAL_NORM)["WER"],
                 "WER sans hésitations": metrics.wer_cer(r, h, text.EVAL_NORM_NOHES)["WER"],
                 "WER lexique majoritaire": metrics.wer_cer([lexicon_normalize(x) for x in r], [lexicon_normalize(x) for x in h], text.RAW_NORM)["WER"],
                 "WER script-aware": metrics.script_aware_wer(r, h)["WER_script_aware"]})
asr_impact = pd.DataFrame(rows).round(2)
display(asr_impact if len(asr_impact) else "Pas encore de prédictions ASR (notebook 01).")

### 6.3 Impact sur la traduction
NLLB zero-shot traduit les transcriptions de référence du test, brutes puis normalisées par les règles.

In [ ]:
mt_impact = None
if RUN_MT_IMPACT:
    from common import translate as T
    tok, mt = T.load_nllb(BASE_MODELS["nllb-600m"])
    out = {}
    for name, fn in [("brut", lambda t: t), ("règles", lambda t: text.normalize_text(t, RULES))]:
        hyp = T.nllb_translate(tok, mt, [fn(t) for t in test.transcription], "eng_Latn")
        out[name] = metrics.translation_scores(hyp, test.translation.tolist())
    mt_impact = pd.DataFrame(out).T[["BLEU", "chrF++"]].round(2); display(mt_impact)
    del mt; torch.cuda.empty_cache()

### 6.4 Analyse manuelle structurée
100 segments du test, une colonne par méthode. Grille pour chaque sortie :
**C** conventions respectées (0/1), **S** sens et mots conservés (0/1), **R** sur-normalisation, c.-à-d. une
distinction utile effacée (0/1). Le fichier `results/03_manual_annotation.csv` est à remplir par l'équipe ;
la cellule suivante agrège les notes une fois le fichier complété.

In [ ]:
man = pd.DataFrame({"source": sample_eval, "règles": [text.normalize_text(t, RULES) for t in sample_eval],
                    "lexique": [lexicon_normalize(t) for t in sample_eval]})
if llm_outputs is not None:
    man["LLM"] = llm_outputs
if (BYT5_DIR / "final").exists() and HAS_GPU and BYT5_OK:
    man["ByT5"] = byt5_normalize(sample_eval)
for m in [c for c in man.columns if c != "source"]:
    for k in ["C", "S", "R"]:
        man[f"{m}_{k}"] = ""
path = RESULTS_DIR / "03_manual_annotation.csv"
if not path.exists():
    man.to_csv(path, index=False)
filled = pd.read_csv(path)
score_cols = [c for c in filled.columns if c.endswith(("_C", "_S", "_R"))]
if filled[score_cols].notna().any().any() and pd.to_numeric(filled[score_cols].stack(), errors="coerce").notna().any():
    display(filled[score_cols].apply(pd.to_numeric, errors="coerce").mean().round(2).rename("score moyen"))
display(man.head(6))

## 7. Vers l'arabe standard (seconde convention de l'énoncé)
NLLB `aeb_Arab → arb_Arab`. Ce n'est plus une normalisation orthographique mais une traduction : le sens est
en principe conservé, mais les emprunts, les marques d'oralité et le registre disparaissent.

In [ ]:
if RUN_MSA:
    from common import translate as T
    tok, mt = T.load_nllb(BASE_MODELS["nllb-600m"])
    s = test.sample(10, random_state=SEED)
    s = s.assign(msa=T.nllb_translate(tok, mt, s.transcription.tolist(), "arb_Arab"))
    display(s[["transcription", "msa", "translation"]])
    del mt; torch.cuda.empty_cache()

## 8. Résultats et limites

In [ ]:
res_rows = [{"évaluation": "réduction du vocabulaire du train (règles, %)", "valeur": float(abl["réduction vocab. (%)"].iloc[-1])}]
res_rows += [{"évaluation": f"exactitude mots inconnus · {k}", "valeur": float(v)} for k, v in acc.items()]
res_rows += [{"évaluation": f"OOV dev+test · {r['méthode']}", "valeur": float(r["OOV (%)"])} for _, r in intr.iterrows()]
# Système ASR rapporté : celui retenu sur le DEV par le notebook 01 (models/asr_best/SOURCE.txt), jamais choisi sur le test
_src = MODELS_DIR / "asr_best" / "SOURCE.txt"
_best_name = _src.read_text().strip() if _src.exists() else None
if len(asr_impact) and _best_name in set(asr_impact["système"]):
    best_sys = asr_impact[asr_impact["système"] == _best_name].iloc[0]
    res_rows += [{"évaluation": f"{c} · {best_sys['système']}", "valeur": float(best_sys[c])} for c in asr_impact.columns[1:]]
if mt_impact is not None:
    res_rows += [{"évaluation": f"chrF++ NLLB · entrée {i}", "valeur": float(r["chrF++"])} for i, r in mt_impact.iterrows()]
norm_res = pd.DataFrame(res_rows).round(2); norm_res.to_csv(RESULTS_DIR / "normalization_results.csv", index=False); display(norm_res)

**Limites d'une normalisation sans orthographe de référence**
- Toute convention est un **choix** : supprimer la hamza simplifie mais efface parfois une distinction
  (أكل « il a mangé » / اكل impératif selon les graphies). Nous l'assumons pour l'évaluation, pas pour la production de texte.
- La convention « forme majoritaire » dépend du corpus : 25 h de TEDx ne reflètent pas tous les usages,
  et la majorité peut changer avec plus de données.
- Le lexique d'emprunts arabe ↔ latin est construit automatiquement puis doit être validé à la main : le squelette
  phonétique produit des faux positifs pour les mots courts.
- Pas de référence normalisée humaine : l'exactitude ByT5 est mesurée sur des variantes **observées** mais
  réassemblées automatiquement ; l'analyse manuelle complète cette évaluation.
- La transformation vers l'arabe standard est une traduction, évaluée seulement qualitativement.

**Améliorations envisagées** : faire annoter 300 phrases selon CODA-TUN (Zribi et al., 2014) pour une vraie
référence ; entraîner ByT5 sur cette référence ; intégrer la normalisation comme pré-traitement des cibles ASR
et mesurer l'effet sur l'entraînement.

## Références
- I. Zribi et al. *A Conventional Orthography for Tunisian Arabic* (CODA-TUN), LREC 2014.
- N. Habash, M. Diab, O. Rambow. *Conventional Orthography for Dialectal Arabic* (CODA), LREC 2012.
- L. Xue et al. *ByT5: Towards a Token-Free Future with Pre-trained Byte-to-Byte Models*, TACL 2022.
- NLLB Team, 2022. Qwen Team, 2024. F. Bougares et al., *TEDxTN*, 2025.